In [ ]:
import pandas as pd

#config
tesla_file = "../../raw/tesla_raw.csv"
output_file = "../../processed/tesla_test.csv"


In [7]:
#load
with open(tesla_file, "r") as f:
    first_line = f.readline()
sep = "\t" if "\t" in first_line else ","

df = pd.read_csv(tesla_file, sep=sep, dtype=str)
df.head()

,peptide,target_value,allele
0,NILGFTFDI,0,HLA-A02:01
1,GAFMRPITLK,0,HLA-A11:01
2,AEVSVLYTV,0,HLA-B44:02
3,FSDQAEVSVL,0,HLA-C05:01
4,ASFCGAVFCKY,0,HLA-B15:01


In [9]:
print(len(df.index))
print(list(df.columns))

714
['peptide', 'target_value', 'allele']


In [12]:
#REFORMATTING
tesla = pd.DataFrame()
tesla["neoantigen_seq"] = df["peptide"].str.strip().str.upper()

#convert 0/1 to Negative/Positive
tesla["label"] = df["target_value"].map({"0": "Negative", "1": "Positive"})

#HLA
tesla["hla_allele"] = df["allele"].str.strip()

tesla.head() 

tesla.head() 

,neoantigen_seq,label,hla_allele
0,NILGFTFDI,Negative,HLA-A02:01
1,GAFMRPITLK,Negative,HLA-A11:01
2,AEVSVLYTV,Negative,HLA-B44:02
3,FSDQAEVSVL,Negative,HLA-C05:01
4,ASFCGAVFCKY,Negative,HLA-B15:01


In [14]:
#quality checks
valid_aa = set("ACDEFGHIKLMNPQRSTVWY")
print("before:", len(tesla.index))

#remove invalid aa
tesla = tesla[tesla["neoantigen_seq"].apply(lambda x: all(aa in valid_aa for aa in x))]
print("after:", len(tesla.index))

before: 714
after: 714


In [15]:
#summary
n_pos = (tesla["label"] == "Positive").sum()
n_neg = (tesla["label"] == "Negative").sum()

print("Total:", len(tesla.index))
print("Positive:", n_pos)
print("Negative:", n_neg)

Total: 714
Positive: 33
Negative: 681


In [16]:
#HLA breakdown
print("HLA alleles present:", tesla["hla_allele"].unique())
print("Top 5 HLA alleles:")
for allele, count in tesla["hla_allele"].value_counts().head(5).items():
    print(f"{allele}: {count}")
print() 

HLA alleles present: ['HLA-A02:01' 'HLA-A11:01' 'HLA-B44:02' 'HLA-C05:01' 'HLA-B15:01'
 'HLA-A23:01' 'HLA-A03:01' 'HLA-A01:01' 'HLA-B27:05' 'HLA-B08:01'
 'HLA-A26:01' 'HLA-B07:02' 'HLA-A24:02' 'HLA-A68:01' 'HLA-C06:02'
 'HLA-B57:01']
Top 5 HLA alleles:
HLA-A02:01: 203
HLA-A01:01: 87
HLA-A23:01: 86
HLA-B08:01: 74
HLA-A03:01: 58



In [17]:
#save
tesla.to_csv(output_file, index=False)